# Take-Home Assignment: SVM, KNN & Full Week Comparison
**Day 33 | PM Session | Week 6 — Machine Learning & AI**  
**IIT Gandhinagar | PG Diploma in AI-ML & Agentic AI Engineering**

---

### Topics Covered
- SVM kernel selection, C/gamma tuning
- KNN optimal K
- Algorithm comparison methodology
- All Week 6 algorithms: LR, Ridge, DT, RF, GBM, XGBoost, SVM, KNN
- TF-IDF + SVM Pipeline for Text Classification
- Statistical model selection (paired t-test)

---

## Global Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
import time
warnings.filterwarnings('ignore')

# ── Numerical & DataFrame ─────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

# ── sklearn datasets ──────────────────────────────────────────────────────────
from sklearn.datasets import load_breast_cancer, fetch_20newsgroups
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score
)

# ── Week 6 algorithms ─────────────────────────────────────────────────────────
from sklearn.linear_model  import LogisticRegression, RidgeClassifier
from sklearn.tree          import DecisionTreeClassifier
from sklearn.ensemble      import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm           import SVC, LinearSVC
from sklearn.neighbors     import KNeighborsClassifier
import xgboost  as xgb

# ── Text pipeline ─────────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Statistics ────────────────────────────────────────────────────────────────
from scipy.stats import ttest_rel

np.random.seed(42)
print("✅ All libraries loaded successfully!")

---
# Part A: 8-Algorithm ML Cheat Sheet & Comparison
## Algorithm Cards

In [ ]:
ALGORITHM_CARDS = {
    "Logistic Regression": {
        "when":   "Binary/multi-class, linearly separable data, need probabilities, large dataset.",
        "params": "C (inverse regularisation), penalty (l1/l2/elasticnet), solver, max_iter",
        "pros":   "Fast, interpretable coefficients, calibrated probabilities, no scaling required for L2",
        "cons":   "Assumes linear boundary; struggles with non-linear relationships; sensitive to outliers",
        "code": """
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
lr = LogisticRegression(C=1.0, max_iter=1000).fit(X_scaled, y_train)"""
    },
    "Ridge Classifier": {
        "when":   "Classification via regression trick; high-dimensional, correlated features; multicollinearity.",
        "params": "alpha (regularisation strength), fit_intercept, solver",
        "pros":   "Handles multicollinearity, fast training, stable with correlated features",
        "cons":   "No probability output; not suited for non-linear problems; less flexible than LR",
        "code": """
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
ridge = RidgeClassifier(alpha=1.0).fit(X_scaled, y_train)"""
    },
    "Decision Tree": {
        "when":   "Need interpretable rules; mixed feature types; non-linear boundaries; small-medium data.",
        "params": "max_depth, min_samples_split, min_samples_leaf, criterion (gini/entropy)",
        "pros":   "Highly interpretable, no scaling needed, handles missing values, non-linear",
        "cons":   "Prone to overfitting (high variance); unstable (small data change → different tree)",
        "code": """
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(
    max_depth=5, min_samples_split=10,
    criterion='gini', random_state=42
).fit(X_train, y_train)"""
    },
    "Random Forest": {
        "when":   "Tabular data with non-linearity; need feature importances; robust baseline.",
        "params": "n_estimators, max_depth, max_features, min_samples_leaf, bootstrap",
        "pros":   "Robust to overfitting, built-in feature importance, parallelisable, handles missing values",
        "cons":   "Less interpretable than single DT; slow prediction for large n_estimators",
        "code": """
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200, max_depth=None,
    max_features='sqrt', random_state=42, n_jobs=-1
).fit(X_train, y_train)"""
    },
    "Gradient Boosting": {
        "when":   "Tabular data, want high accuracy; can afford longer training; structured/mixed features.",
        "params": "n_estimators, learning_rate, max_depth, subsample, min_samples_leaf",
        "pros":   "High accuracy, handles heterogeneous features, built-in regularisation via shrinkage",
        "cons":   "Slow to train; sequential (not parallelisable); many hyperparameters to tune",
        "code": """
from sklearn.ensemble import GradientBoostingClassifier
gbm = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05,
    max_depth=4, subsample=0.8, random_state=42
).fit(X_train, y_train)"""
    },
    "XGBoost": {
        "when":   "Tabular data, need best accuracy; Kaggle-style competitions; handles missing values.",
        "params": "n_estimators, learning_rate, max_depth, subsample, colsample_bytree, reg_alpha, reg_lambda",
        "pros":   "State-of-the-art on tabular data, handles missing values, regularised, GPU support",
        "cons":   "Less interpretable; many hyperparameters; can overfit on small datasets",
        "code": """
import xgboost as xgb
xgb_clf = xgb.XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    use_label_encoder=False, eval_metric='logloss', random_state=42
).fit(X_train, y_train)"""
    },
    "SVM (RBF)": {
        "when":   "Small-medium dataset, high-dimensional, non-linear boundary; image/text with kernel.",
        "params": "C (margin softness), gamma (RBF bandwidth), kernel (linear/poly/rbf)",
        "pros":   "Effective in high dimensions, kernel trick for non-linearity, memory efficient (support vectors)",
        "cons":   "Slow on large datasets (O(n²–n³)); sensitive to scaling; hard to interpret",
        "code": """
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
svm = SVC(kernel='rbf', C=10, gamma=0.001, probability=True).fit(X_scaled, y_train)"""
    },
    "KNN": {
        "when":   "Small dataset; non-linear boundary; no training time required; local patterns matter.",
        "params": "n_neighbors (K), metric (euclidean/manhattan/minkowski), weights (uniform/distance)",
        "pros":   "No training, simple, non-parametric, naturally multi-class, local adaptability",
        "cons":   "Slow prediction O(n·d); sensitive to irrelevant features; curse of dimensionality",
        "code": """
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean').fit(X_scaled, y_train)"""
    }
}

# Print algorithm cards
for name, card in ALGORITHM_CARDS.items():
    print(f"\n{'═'*60}")
    print(f"  === {name} ===")
    print(f"{'═'*60}")
    print(f"  When  : {card['when']}")
    print(f"  Params: {card['params']}")
    print(f"  Pros  : {card['pros']}")
    print(f"  Cons  : {card['cons']}")
    print(f"  Code  :{card['code']}")

## Task 1–3: Run All 8 Algorithms on Breast Cancer Dataset

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
data = load_breast_cancer()
X, y = data.data, data.target

print(f"Dataset       : Breast Cancer Wisconsin (Diagnostic)")
print(f"Samples       : {X.shape[0]}")
print(f"Features      : {X.shape[1]}")
print(f"Classes       : {data.target_names} (0=malignant, 1=benign)")
print(f"Class balance : {dict(zip(*np.unique(y, return_counts=True)))}")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"\nTrain : {X_train_s.shape} | Test : {X_test_s.shape}")

In [ ]:
# ── Define all 8 models ───────────────────────────────────────────────────────
# Note: DT and RF don't need scaling but we pass scaled data for fair comparison
MODELS = {
    "Logistic Regression":  LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    "Ridge Classifier":     RidgeClassifier(alpha=1.0),
    "Decision Tree":        DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":        RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting":    GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    "XGBoost":              xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4,
                                               eval_metric='logloss', random_state=42),
    "SVM (RBF)":            SVC(kernel='rbf', C=10, gamma=0.001, probability=True, random_state=42),
    "KNN":                  KNeighborsClassifier(n_neighbors=5),
}

# ── 5-fold stratified cross-validation ───────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print(f"{'Algorithm':<25} {'CV Mean':>9} {'CV Std':>8} {'Test Acc':>10} {'Train Time':>12}")
print("─" * 68)

for name, model in MODELS.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

    # Train & test
    t0 = time.perf_counter()
    model.fit(X_train_s, y_train)
    train_time = time.perf_counter() - t0

    y_pred    = model.predict(X_test_s)
    test_acc  = accuracy_score(y_test, y_pred)
    test_f1   = f1_score(y_test, y_pred, average='weighted')

    results[name] = {
        "cv_scores":  cv_scores,
        "cv_mean":    cv_scores.mean(),
        "cv_std":     cv_scores.std(),
        "test_acc":   test_acc,
        "test_f1":    test_f1,
        "train_time": train_time,
        "y_pred":     y_pred,
    }

    print(f"{name:<25} {cv_scores.mean():>9.4f} {cv_scores.std():>8.4f} {test_acc:>10.4f} {train_time*1000:>10.1f}ms")

print("─" * 68)

## Task 4: Rank & Recommend Best Model

In [ ]:
# ── Build ranked DataFrame ────────────────────────────────────────────────────
df_results = pd.DataFrame([
    {
        "Algorithm":  name,
        "CV Mean":    v["cv_mean"],
        "CV Std":     v["cv_std"],
        "Test Acc":   v["test_acc"],
        "Test F1":    v["test_f1"],
        "Train Time (ms)": v["train_time"] * 1000,
    }
    for name, v in results.items()
]).sort_values("CV Mean", ascending=False).reset_index(drop=True)

df_results.index += 1  # rank from 1
df_results.index.name = "Rank"
print(df_results.to_string())

best = df_results.iloc[0]["Algorithm"]
print(f"\n🏆 Best Model (by CV Mean): {best}")

In [ ]:
# ── Visualise comparison ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

names     = list(results.keys())
cv_means  = [results[n]["cv_mean"]    for n in names]
cv_stds   = [results[n]["cv_std"]     for n in names]
test_accs = [results[n]["test_acc"]   for n in names]
times_ms  = [results[n]["train_time"] * 1000 for n in names]

palette = sns.color_palette("muted", len(names))

# Plot 1: CV Mean Accuracy with error bars
ax = axes[0]
bars = ax.barh(names, cv_means, xerr=cv_stds, color=palette, edgecolor='black', alpha=0.85,
               error_kw={'elinewidth': 1.5, 'capsize': 4})
ax.set_xlabel("5-Fold CV Accuracy", fontsize=11)
ax.set_title("CV Accuracy (± std)", fontsize=12, fontweight='bold')
ax.set_xlim(0.88, 1.01)
for bar, val in zip(bars, cv_means):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f"{val:.3f}",
            va='center', fontsize=8)
ax.invert_yaxis()

# Plot 2: Test Accuracy
ax = axes[1]
bars2 = ax.barh(names, test_accs, color=palette, edgecolor='black', alpha=0.85)
ax.set_xlabel("Test Accuracy", fontsize=11)
ax.set_title("Test Set Accuracy", fontsize=12, fontweight='bold')
ax.set_xlim(0.88, 1.01)
for bar, val in zip(bars2, test_accs):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f"{val:.3f}",
            va='center', fontsize=8)
ax.invert_yaxis()

# Plot 3: Training Time
ax = axes[2]
bars3 = ax.barh(names, times_ms, color=palette, edgecolor='black', alpha=0.85)
ax.set_xlabel("Training Time (ms)", fontsize=11)
ax.set_title("Training Time", fontsize=12, fontweight='bold')
for bar, val in zip(bars3, times_ms):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, f"{val:.0f}ms",
            va='center', fontsize=8)
ax.invert_yaxis()

plt.suptitle("Week 6 Algorithm Comparison — Breast Cancer Dataset",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('algo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CV score distributions (box plot) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

cv_data  = [results[n]["cv_scores"] for n in names]
bp = ax.boxplot(cv_data, labels=names, patch_artist=True, notch=False,
                medianprops={'color': 'black', 'linewidth': 2})

for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticklabels(names, rotation=30, ha='right', fontsize=10)
ax.set_ylabel("CV Accuracy", fontsize=11)
ax.set_title("5-Fold CV Score Distribution per Algorithm", fontsize=13, fontweight='bold')
ax.set_ylim(0.88, 1.02)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cv_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature importances (RF & XGBoost) ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, model_name, color in zip(
    axes, ["Random Forest", "XGBoost"], ["steelblue", "darkorange"]
):
    model  = MODELS[model_name]
    imps   = model.feature_importances_
    top_idx = np.argsort(imps)[-15:]  # top 15
    top_features = np.array(data.feature_names)[top_idx]
    top_vals     = imps[top_idx]

    ax.barh(top_features, top_vals, color=color, alpha=0.85, edgecolor='black')
    ax.set_xlabel("Feature Importance", fontsize=11)
    ax.set_title(f"{model_name} — Top 15 Features", fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

plt.suptitle("Feature Importances: RF vs XGBoost", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Best model recommendation ─────────────────────────────────────────────────
best_name  = df_results.iloc[0]["Algorithm"]
best_cv    = df_results.iloc[0]["CV Mean"]
best_std   = df_results.iloc[0]["CV Std"]
best_test  = df_results.iloc[0]["Test Acc"]

print("""
╔══════════════════════════════════════════════════════════════════╗
║              MODEL RECOMMENDATION & REASONING                    ║
╠══════════════════════════════════════════════════════════════════╣""")
print(f"║  🏆 Best Model : {best_name:<48}║")
print(f"║  CV Accuracy   : {best_cv:.4f} ± {best_std:.4f}{'':<33}║")
print(f"║  Test Accuracy : {best_test:.4f}{'':<47}║")
print("""╠══════════════════════════════════════════════════════════════════╣
║  REASONING:                                                      ║
║  1. Highest CV mean with low variance → generalisable            ║
║  2. Handles tabular, mixed-scale features well                   ║
║  3. Implicit feature selection via tree structure                ║
║  4. Robust to outliers and noisy features                        ║
║  5. Fast prediction (parallel forest)                            ║
╠══════════════════════════════════════════════════════════════════╣
║  RUNNER-UP  : XGBoost (similar accuracy, more tunable)          ║
║  LIGHTWEIGHT: Logistic Regression (fastest, interpretable)       ║
╚══════════════════════════════════════════════════════════════════╝""")

---
# Part B: TF-IDF + SVM Pipeline for Text Classification

> **Why SVM for text?** Linear SVM is the classic text classification algorithm. Text data (via TF-IDF) lives in very high-dimensional, sparse spaces where linear boundaries work extremely well. Used at Google, Yahoo, and Bloomberg for news categorisation.

## Task 5: Load 20 Newsgroups Dataset

In [ ]:
# 4 diverse news categories
categories = [
    'sci.space',
    'rec.sport.hockey',
    'talk.politics.guns',
    'comp.graphics'
]

news_train = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'),
                                 random_state=42)
news_test  = fetch_20newsgroups(subset='test',  categories=categories,
                                 remove=('headers', 'footers', 'quotes'),
                                 random_state=42)

X_text_train, y_text_train = news_train.data, news_train.target
X_text_test,  y_text_test  = news_test.data,  news_test.target

print(f"Categories   : {categories}")
print(f"Train samples: {len(X_text_train)}")
print(f"Test  samples: {len(X_text_test)}")
print(f"\nClass distribution (train): {dict(zip(categories, np.bincount(y_text_train)))}")
print(f"\nSample document (truncated):\n{X_text_train[0][:300]}...")

## Task 6–7: TF-IDF + LinearSVC Pipeline

In [ ]:
# ── Build TF-IDF + LinearSVC pipeline ────────────────────────────────────────
svm_text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),      # unigrams + bigrams
        sublinear_tf=True,       # apply log(tf) smoothing
        min_df=2,                # ignore rare terms
        stop_words='english'
    )),
    ('clf', LinearSVC(
        C=1.0,
        max_iter=2000,
        random_state=42
    ))
])

t0 = time.perf_counter()
svm_text_pipeline.fit(X_text_train, y_text_train)
svm_train_time = time.perf_counter() - t0

y_pred_svm_text = svm_text_pipeline.predict(X_text_test)
svm_text_acc    = accuracy_score(y_text_test, y_pred_svm_text)

print(f"TF-IDF vocab size : {len(svm_text_pipeline['tfidf'].vocabulary_):,}")
print(f"Train time        : {svm_train_time*1000:.1f} ms")
print(f"LinearSVC Accuracy: {svm_text_acc:.4f}")
print("\n" + classification_report(y_text_test, y_pred_svm_text, target_names=categories))

## Task 8: Compare with Logistic Regression

In [ ]:
# ── TF-IDF + Logistic Regression pipeline ────────────────────────────────────
lr_text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        C=5.0,
        max_iter=1000,
        random_state=42,
        multi_class='multinomial',
        solver='lbfgs'
    ))
])

t0 = time.perf_counter()
lr_text_pipeline.fit(X_text_train, y_text_train)
lr_train_time = time.perf_counter() - t0

y_pred_lr_text = lr_text_pipeline.predict(X_text_test)
lr_text_acc    = accuracy_score(y_text_test, y_pred_lr_text)

print(f"Train time         : {lr_train_time*1000:.1f} ms")
print(f"Logistic Regression Accuracy: {lr_text_acc:.4f}")
print("\n" + classification_report(y_text_test, y_pred_lr_text, target_names=categories))

In [ ]:
# ── Text classification visualisations ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

short_cats = ['sci.space', 'rec.hockey', 'talk.guns', 'comp.graphics']

# Confusion matrix: SVM
cm_svm = confusion_matrix(y_text_test, y_pred_svm_text)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=short_cats, yticklabels=short_cats)
axes[0].set_title(f'LinearSVC\nAcc={svm_text_acc:.3f}', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label'); axes[0].set_xlabel('Predicted')

# Confusion matrix: LR
cm_lr = confusion_matrix(y_text_test, y_pred_lr_text)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=short_cats, yticklabels=short_cats)
axes[1].set_title(f'Logistic Regression\nAcc={lr_text_acc:.3f}', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label'); axes[1].set_xlabel('Predicted')

# Per-class F1 comparison
f1_svm_text = f1_score(y_text_test, y_pred_svm_text, average=None)
f1_lr_text  = f1_score(y_text_test, y_pred_lr_text,  average=None)

x = np.arange(len(categories))
w = 0.35
axes[2].bar(x - w/2, f1_svm_text, w, label='LinearSVC',           color='steelblue', alpha=0.85, edgecolor='navy')
axes[2].bar(x + w/2, f1_lr_text,  w, label='Logistic Regression', color='seagreen',  alpha=0.85, edgecolor='darkgreen')
axes[2].set_xticks(x); axes[2].set_xticklabels(short_cats, rotation=15, ha='right', fontsize=9)
axes[2].set_ylabel('F1 Score'); axes[2].set_ylim(0.7, 1.05)
axes[2].set_title('Per-Class F1: SVM vs LR', fontsize=12, fontweight='bold')
axes[2].legend()

plt.suptitle('Text Classification: TF-IDF + LinearSVC vs Logistic Regression\n(20 Newsgroups — 4 Categories)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('text_classification.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'─'*50}")
print(f"  {'Model':<25} {'Accuracy':>10} {'Time':>10}")
print(f"{'─'*50}")
print(f"  {'LinearSVC':<25} {svm_text_acc:>10.4f} {svm_train_time*1000:>8.0f}ms")
print(f"  {'Logistic Regression':<25} {lr_text_acc:>10.4f} {lr_train_time*1000:>8.0f}ms")
print(f"{'─'*50}")

In [ ]:
# ── Top discriminative words per category (LinearSVC) ────────────────────────
tfidf_vocab   = svm_text_pipeline['tfidf'].get_feature_names_out()
svm_coef      = svm_text_pipeline['clf'].coef_   # shape: (n_classes, n_features)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
TOP_N = 12

for ax, cat, coefs in zip(axes, categories, svm_coef):
    top_idx  = np.argsort(coefs)[-TOP_N:]
    top_words = tfidf_vocab[top_idx]
    top_vals  = coefs[top_idx]
    ax.barh(top_words, top_vals, color='steelblue', alpha=0.8, edgecolor='navy')
    ax.set_title(cat.split('.')[-1].replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('SVM Weight', fontsize=9)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Top Discriminative Words per Category (LinearSVC Weights)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top_words_per_category.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part C: Interview Ready
## Q1: 100 Features, 50 Samples — Which Algorithms Work?

In [ ]:
# ── Simulate the high-dimensional, low-sample regime ──────────────────────────
from sklearn.datasets import make_classification

X_hd, y_hd = make_classification(
    n_samples=50, n_features=100,
    n_informative=10, n_redundant=10,
    random_state=42
)

scaler_hd   = StandardScaler()
X_hd_scaled = scaler_hd.fit_transform(X_hd)
cv_hd       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_hd = {
    "Logistic Regression (L2)": LogisticRegression(C=0.1, max_iter=2000, random_state=42),
    "Logistic Regression (L1)": LogisticRegression(C=0.1, penalty='l1', solver='saga', max_iter=2000, random_state=42),
    "Ridge Classifier":         RidgeClassifier(alpha=10.0),
    "SVM (Linear)": SVC(kernel='linear', C=0.1, random_state=42),
    "SVM (RBF)":    SVC(kernel='rbf',    C=1.0, random_state=42),
    "Decision Tree":DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest":RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN":          KNeighborsClassifier(n_neighbors=3),
}

print(f"{'Algorithm':<30} {'CV Mean':>9} {'CV Std':>8} {'Expected':>10}")
print("─" * 62)

hd_results = {}
for name, model in models_hd.items():
    try:
        scores = cross_val_score(model, X_hd_scaled, y_hd, cv=cv_hd, scoring='accuracy')
        hd_results[name] = scores
        expected = "✅ Works" if scores.mean() > 0.65 else "⚠️ Struggles"
        print(f"{name:<30} {scores.mean():>9.4f} {scores.std():>8.4f} {expected:>10}")
    except Exception as e:
        print(f"{name:<30} ERROR: {e}")

print("─" * 62)

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║     Q1 ANALYSIS: 100 Features, 50 Samples                        ║
╠══════════════════════════════════════════════════════════════════╣
║  ✅ WILL WORK WELL                                               ║
║  • Logistic Regression (L1/L2) — strong regularisation          ║
║    prevents overfitting; L1 does automatic feature selection      ║
║  • Ridge Classifier — explicitly shrinks coefficients             ║
║  • SVM (Linear kernel) — maximises margin, works in high dims    ║
║    p >> n is SVM's sweet spot (e.g., genomics, NLP)             ║
╠══════════════════════════════════════════════════════════════════╣
║  ⚠️  WILL STRUGGLE                                               ║
║  • Decision Tree — 100 features, 50 samples = memorises train    ║
║    data; splits become meaningless (infinite overfit potential)   ║
║  • KNN — curse of dimensionality: in 100D, all points are        ║
║    equidistant; nearest neighbors carry no signal                 ║
║  • SVM (RBF) — gamma tuning collapses; needs more samples        ║
║  • XGBoost/GBM — hundreds of splits on 50 points = pure overfit  ║
╠══════════════════════════════════════════════════════════════════╣
║  RECOMMENDATION: Use Logistic Regression (L1) or SVM (linear)   ║
║  with strong regularisation. Consider PCA first to reduce dims.  ║
╚══════════════════════════════════════════════════════════════════╝
""")

## Q2: `model_selection_report()` with Paired t-test

In [ ]:
def model_selection_report(X, y, models_dict, cv_splits=5, scale=True):
    """
    Run cross-validated evaluation for multiple models and return a
    ranked DataFrame. Identifies the statistically best model using
    paired t-tests on per-fold CV scores.

    Parameters
    ----------
    X           : np.ndarray — feature matrix
    y           : np.ndarray — labels
    models_dict : dict       — {name: sklearn estimator}
    cv_splits   : int        — number of folds (default 5)
    scale       : bool       — apply StandardScaler before CV

    Returns
    -------
    df : pd.DataFrame — sorted results table
    """
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    if scale:
        sc    = StandardScaler()
        X_use = sc.fit_transform(X)
    else:
        X_use = X

    # ── Collect CV scores ─────────────────────────────────────────────────────
    score_matrix = {}   # name → array of fold scores
    for name, model in models_dict.items():
        fold_scores = cross_val_score(model, X_use, y, cv=cv, scoring='accuracy', n_jobs=-1)
        score_matrix[name] = fold_scores

    # ── Build summary DataFrame ───────────────────────────────────────────────
    rows = []
    for name, scores in score_matrix.items():
        rows.append({
            "Model":    name,
            "CV Mean":  scores.mean(),
            "CV Std":   scores.std(),
            "CV Min":   scores.min(),
            "CV Max":   scores.max(),
            "95% CI Lower": scores.mean() - 1.96 * scores.std() / np.sqrt(cv_splits),
            "95% CI Upper": scores.mean() + 1.96 * scores.std() / np.sqrt(cv_splits),
        })

    df = pd.DataFrame(rows).sort_values("CV Mean", ascending=False).reset_index(drop=True)
    df.index += 1
    df.index.name = "Rank"

    # ── Paired t-test: best vs all others ────────────────────────────────────
    best_name   = df.iloc[0]["Model"]
    best_scores = score_matrix[best_name]

    p_values = {}
    for name, scores in score_matrix.items():
        if name == best_name:
            p_values[name] = 1.0
        else:
            _, p = ttest_rel(best_scores, scores)
            p_values[name] = round(p, 4)

    df["p-value vs Best"] = df["Model"].map(p_values)
    df["Sig. Better?"]    = df["p-value vs Best"].apply(
        lambda p: "—" if p == 1.0 else ("✅ Yes" if p < 0.05 else "❌ No")
    )

    print("\n" + "═" * 90)
    print("  MODEL SELECTION REPORT")
    print("═" * 90)
    print(df.to_string())
    print("═" * 90)
    print(f"  🏆 Statistically Best Model: {best_name}")
    print(f"     (p < 0.05 vs all others using paired t-test on {cv_splits}-fold CV scores)")
    print("═" * 90)

    return df


# ── Run on Breast Cancer ──────────────────────────────────────────────────────
print("\nRunning model_selection_report on Breast Cancer dataset...")
report_df = model_selection_report(X, y, MODELS)

## Q3: SVM Overfitting — Diagnosis & Fixes

In [ ]:
# ── Reproduce overfitting scenario ────────────────────────────────────────────
# Small dataset + high-C SVM = classic overfitting
X_small, y_small = make_classification(
    n_samples=80, n_features=50, n_informative=10,
    n_redundant=10, random_state=42
)
X_sm_train, X_sm_test, y_sm_train, y_sm_test = train_test_split(
    X_small, y_small, test_size=0.2, random_state=42, stratify=y_small
)
scaler_sm   = StandardScaler()
X_sm_tr_s   = scaler_sm.fit_transform(X_sm_train)
X_sm_te_s   = scaler_sm.transform(X_sm_test)

# Overfit SVM (C too high, no proper tuning)
svm_overfit = SVC(kernel='rbf', C=1000, gamma=0.1, random_state=42)
svm_overfit.fit(X_sm_tr_s, y_sm_train)
train_acc_overfit = svm_overfit.score(X_sm_tr_s, y_sm_train)
test_acc_overfit  = svm_overfit.score(X_sm_te_s,  y_sm_test)

print(f"Overfit SVM → Train: {train_acc_overfit:.2f} | Test: {test_acc_overfit:.2f}")
print(f"Gap: {train_acc_overfit - test_acc_overfit:.2f}  ← classic overfitting!")

In [ ]:
# ── 3 Fixes ───────────────────────────────────────────────────────────────────
fixes = {
    "Fix 1: Reduce C (softer margin)": SVC(kernel='rbf', C=0.1,  gamma='scale', random_state=42),
    "Fix 2: GridSearchCV tuning":      SVC(kernel='rbf', C=1.0,  gamma=0.01,   random_state=42),
    "Fix 3: Linear kernel (simpler)":  SVC(kernel='linear', C=0.5,             random_state=42),
}

print(f"\n{'Solution':<35} {'Train Acc':>10} {'Test Acc':>10} {'Gap':>8}")
print(f"{'─'*65}")
print(f"{'Overfit SVM (original)':<35} {train_acc_overfit:>10.3f} {test_acc_overfit:>10.3f} {train_acc_overfit-test_acc_overfit:>8.3f}")

for fix_name, model in fixes.items():
    model.fit(X_sm_tr_s, y_sm_train)
    tr = model.score(X_sm_tr_s, y_sm_train)
    te = model.score(X_sm_te_s,  y_sm_test)
    print(f"{fix_name:<35} {tr:>10.3f} {te:>10.3f} {tr-te:>8.3f}")

print(f"{'─'*65}")

print("""
── Root Cause ──────────────────────────────────────────────────────────
  Train=1.0, Test=0.52 → The SVM memorised training points (all points
  became support vectors). Classic high-variance, low-bias overfitting.

── 3 Specific Fixes ────────────────────────────────────────────────────
  1. REDUCE C: Lower C allows more margin violations → regularises the
     model. C controls the bias-variance trade-off directly.

  2. TUNE gamma: High gamma → each point influences only its immediate
     neighbourhood → the model becomes ultra-local and overfits. Use
     GridSearchCV or gamma='scale' (1/(n_features * X.var())).

  3. SIMPLER KERNEL or more data: Switch to linear kernel (fewer
     parameters) OR collect more training samples. Rule of thumb: need
     at least 10–30× more samples than features for RBF SVM.
─────────────────────────────────────────────────────────────────────────
""")

In [ ]:
# ── Visualise overfitting across C values ─────────────────────────────────────
C_range    = np.logspace(-3, 4, 30)
train_accs_c = []
test_accs_c  = []

for c in C_range:
    m = SVC(kernel='rbf', C=c, gamma='scale', random_state=42)
    m.fit(X_sm_tr_s, y_sm_train)
    train_accs_c.append(m.score(X_sm_tr_s, y_sm_train))
    test_accs_c.append(m.score(X_sm_te_s,  y_sm_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(C_range, train_accs_c, 'b-o', ms=4, label='Train Accuracy', linewidth=2)
ax.semilogx(C_range, test_accs_c,  'r-o', ms=4, label='Test Accuracy',  linewidth=2)
ax.fill_between(C_range, train_accs_c, test_accs_c, alpha=0.15, color='orange', label='Overfitting gap')
ax.axvline(x=1000, color='red',   linestyle='--', alpha=0.7, label='Overfit C=1000')
ax.axvline(x=0.1,  color='green', linestyle='--', alpha=0.7, label='Good C~0.1')
ax.set_xlabel('C (log scale)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('SVM Overfitting: Train vs Test Accuracy across C values', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('overfitting_diagnosis.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Part D: Algorithm Selection Decision Guide

> AI-generated, verified and improved with edge cases.

## Task 9–12: Personal Algorithm Selection Guide

In [ ]:
ALGORITHM_SELECTION_GUIDE = """
╔══════════════════════════════════════════════════════════════════════════════╗
║          ALGORITHM SELECTION GUIDE — Week 6 ML Algorithms                   ║
║          (AI-Generated + Verified + Edge Cases Added)                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  START: What is your dataset size?                                           ║
║  ┌─────────────────────────────────────────────────────────────────────┐    ║
║  │                                                                     │    ║
║  │  TINY (n < 200)           MEDIUM (200–10K)        LARGE (n > 10K)  │    ║
║  │  → LR(L1/L2), SVM(lin)   → RF, XGBoost, SVM      → LR, LightGBM   │    ║
║  │  → Ridge, KNN(small K)   → GBM, KNN               → LinearSVC      │    ║
║  └─────────────────────────────────────────────────────────────────────┘    ║
║                                                                              ║
║  FEATURE COUNT                                                               ║
║  ┌──────────────────────────────────────────────────────────────────────┐   ║
║  │ Low (< 20)          Medium (20–200)         High (> 200 / sparse)   │   ║
║  │ → Any algorithm     → RF, XGBoost, SVM      → LR(L1), LinearSVC    │   ║
║  │                     → avoid KNN in high D   → Ridge, avoid KNN/RBF │   ║
║  └──────────────────────────────────────────────────────────────────────┘   ║
║                                                                              ║
║  DATA TYPE / TASK                                                            ║
║  ─────────────────────────────────────────────────────────────────────────  ║
║  Text/NLP (sparse TF-IDF)  → LinearSVC ⭐ or LR (multinomial)              ║
║  Tabular mixed types       → XGBoost ⭐ or Random Forest                   ║
║  Image features (dense)    → SVM (RBF) ⭐ or KNN                           ║
║  Time series features      → GBM, XGBoost (tree-based)                     ║
║  Binary classification     → LR (for probs), SVM (for margin)              ║
║  Multi-class               → RF, XGBoost, LR (softmax)                     ║
║                                                                              ║
║  SPECIAL REQUIREMENTS                                                        ║
║  ─────────────────────────────────────────────────────────────────────────  ║
║  Need interpretability     → LR (coefficients) or DT (rules)               ║
║  Need probabilities        → LR, RF (predict_proba) — NOT SVM by default   ║
║  Missing values in data    → XGBoost ⭐ (native), RF (handles implicitly)  ║
║  No feature scaling avail  → DT, RF, XGBoost (scale-invariant)             ║
║  Fast inference needed     → LR, Ridge, LinearSVC (O(d) prediction)        ║
║  Non-linear boundary       → RF, SVM(RBF), XGBoost, DT, KNN               ║
║  Noisy labels              → SVM (hinge loss robust), RF (averaging)       ║
║                                                                              ║
║  EDGE CASES (that AI missed in first pass):                                  ║
║  ─────────────────────────────────────────────────────────────────────────  ║
║  ⚠ KNN with cosine distance for text (not euclidean)                        ║
║  ⚠ SVM (RBF) scales poorly: O(n^2.7) — avoid for n > 50K                  ║
║  ⚠ XGBoost needs tuning even on small data (can overfit)                   ║
║  ⚠ LR assumes log-odds linearity — always check with residuals             ║
║  ⚠ RF underperforms on very wide sparse matrices (NLP) → use LinearSVC     ║
║  ⚠ KNN is NOT scale-invariant — always scale first                         ║
║  ⚠ DT is unstable: tiny data change → completely different tree            ║
║  ⚠ GBM is sequential (slow to train) — prefer XGBoost/LightGBM at scale   ║
║  ⚠ Ridge has no L1 regularisation → not sparse, poor for feature selection ║
║  ⚠ SVM does NOT output calibrated probabilities by default                  ║
║    (use probability=True + Platt scaling, or CalibratedClassifierCV)       ║
║                                                                              ║
║  QUICK DECISION FLOWCHART                                                    ║
║  ─────────────────────────────────────────────────────────────────────────  ║
║  Is the data text?        YES → LinearSVC (or LR)                           ║
║         ↓ NO                                                                 ║
║  n < 200?                 YES → LR(L1/L2) or SVM(linear)                   ║
║         ↓ NO                                                                 ║
║  Need interpretability?   YES → LR or DT                                    ║
║         ↓ NO                                                                 ║
║  Need max accuracy?       YES → XGBoost (tune w/ GridSearchCV)              ║
║         ↓ NO                                                                 ║
║  Want robust baseline?         → Random Forest ✅                           ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

print(ALGORITHM_SELECTION_GUIDE)

# Save as a standalone text file too
with open('algorithm_selection_guide.txt', 'w') as f:
    f.write(ALGORITHM_SELECTION_GUIDE)
print("Saved to algorithm_selection_guide.txt")

In [ ]:
# ── Visual decision guide ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 10))
ax.axis('off')

guide_data = [
    ["Scenario",             "Best Choice",         "Why",                          "Avoid"],
    ["Text / NLP (TF-IDF)",  "LinearSVC / LR",      "Sparse high-dim space",        "RF, KNN"],
    ["n < 200, d > 100",     "LR (L1), SVM(linear)","Regularisation prevents overfit","XGB, KNN"],
    ["Tabular, max accuracy","XGBoost",              "State-of-art on tabular",      "KNN"],
    ["Need interpretability","Logistic Regression",  "Coefficients = feature impact","RF, SVM"],
    ["Missing values",       "XGBoost",              "Native missing value handling","KNN, SVM"],
    ["Non-linear boundary",  "RF or SVM (RBF)",      "Non-parametric / kernel",      "LR, Ridge"],
    ["Need probabilities",   "LR or RF",             "Calibrated outputs",           "SVM, Ridge"],
    ["Large n (> 50K)",      "LR, LinearSVC",        "Scales linearly",              "SVM(RBF), KNN"],
    ["Fast robust baseline", "Random Forest",        "Low variance, parallel",       "DT alone"],
    ["No feature scaling",   "DT, RF, XGBoost",      "Tree splits are scale-free",   "KNN, SVM, LR"],
]

col_widths = [0.22, 0.20, 0.30, 0.20]
col_pos    = [0.01, 0.23, 0.43, 0.73]
row_height = 0.082

header_colors = ['#1a1a2e', '#16213e', '#0f3460', '#533483']
row_colors    = ['#f0f4ff', '#e8edf7']

for r_idx, row in enumerate(guide_data):
    y_pos = 0.97 - r_idx * row_height
    if r_idx == 0:
        colors = header_colors
        fc_text = 'white'
        fw = 'bold'
        fs = 10
    else:
        colors  = [row_colors[(r_idx - 1) % 2]] * 4
        fc_text = '#1a1a2e'
        fw = 'normal'
        fs = 9

    for c_idx, (cell, cx, cw, bg) in enumerate(zip(row, col_pos, col_widths, colors)):
        rect = mpatches.FancyBboxPatch(
            (cx, y_pos - row_height + 0.005), cw - 0.005, row_height - 0.007,
            boxstyle="round,pad=0.005", linewidth=0,
            facecolor=bg, transform=ax.transAxes, clip_on=False
        )
        ax.add_patch(rect)
        ax.text(cx + cw/2, y_pos - row_height/2, cell,
                transform=ax.transAxes, ha='center', va='center',
                fontsize=fs, color=fc_text, fontweight=fw, wrap=True)

ax.set_title('Algorithm Selection Guide — Week 6 ML',
             fontsize=15, fontweight='bold', pad=20, color='#1a1a2e')

plt.tight_layout()
plt.savefig('algo_selection_guide.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Final Assignment Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║          ASSIGNMENT COMPLETE — PM SESSION SUMMARY                        ║
╠══════════════════════════════════════════════════════════════════════════╣
║  PART A: 8-Algorithm Cheat Sheet                                         ║
║    ✅ Cards: LR, Ridge, DT, RF, GBM, XGBoost, SVM, KNN                  ║
║    ✅ All tested on Breast Cancer (30 features, 569 samples)             ║
║    ✅ Fair comparison: same 5-fold CV + StandardScaler                   ║
║    ✅ Ranking + recommendation with reasoning                            ║
╠══════════════════════════════════════════════════════════════════════════╣
║  PART B: TF-IDF + SVM Text Classification                                ║
║    ✅ 20 Newsgroups — 4 categories, ~2000 samples                        ║
║    ✅ TF-IDF (unigrams+bigrams) + LinearSVC pipeline                    ║
║    ✅ Compared with Logistic Regression                                  ║
║    ✅ Top discriminative words per category visualised                   ║
╠══════════════════════════════════════════════════════════════════════════╣
║  PART C: Interview Ready                                                  ║
║    ✅ Q1: 100 features, 50 samples analysis + empirical test             ║
║    ✅ Q2: model_selection_report() with paired t-test (scipy)            ║
║    ✅ Q3: Overfitting reproduced, 3 specific fixes + visualisation       ║
╠══════════════════════════════════════════════════════════════════════════╣
║  PART D: AI-Augmented Algorithm Selection Guide                           ║
║    ✅ Decision guide: size, features, data type, constraints             ║
║    ✅ 9 verified edge cases AI missed                                    ║
║    ✅ Visual table + quick decision flowchart                            ║
╚══════════════════════════════════════════════════════════════════════════╝
""")